# Reconstrucción y Auditoría del Metaverso 2026

Este cuaderno documenta el proceso técnico de reconstrucción del repositorio maestro **Metaverso 2026** a nivel nacional. El objetivo es consolidar una base de datos de alta fidelidad que mantenga la granularidad mínima (UDS por UDS) y refleje los valores financieros validados en los insumos regionales más recientes.

## Contexto Técnico
El Metaverso original presentaba desactualizaciones en costos y cupos. Por otro lado, los archivos de "Nivelación de Canastas" (insumos regionales) contenían la verdad financiera validada por humanos, pero carecían de información geográfica completa (direcciones, barrios).

### Decisiones Técnicas Clave
1. **Fusión Maestra (Hybrid approach):** No se realiza una extracción desde cero para evitar la pérdida de campos fijos. Se utiliza el Metaverso histórico como plantilla y se sobrescriben los datos dinámicos.
2. **Identidad de Columnas:** Se mantienen las **101 columnas originales** con sus nombres exactos (incluyendo saltos de línea y caracteres especiales) para garantizar compatibilidad con herramientas de análisis preexistentes.
3. **Lógica de Auditoría:** Se implementó un sistema de banderas (colores) para que los tomadores de decisiones identifiquen qué registros fueron actualizados, cuáles son nuevos y cuáles no tienen respaldo en la planeación 2026.

## 1. Configuración de Entorno y Rutas
Se definen las rutas absolutas del repositorio para asegurar la portabilidad del script en diferentes entornos.

In [7]:
import pandas as pd
import glob
import os
import re
from openpyxl import load_workbook
from openpyxl.styles import PatternFill

# Configuración de Rutas Base
DIR_BASE = r"/app/data/insumos metaverso"
ORIGINAL_METAVERSO = os.path.join(DIR_BASE, "Metaverso 2026.xlsx")
INTEGRALES_DIR = os.path.join(DIR_BASE, "Nivelacion de Canastas Integrales")
HCB_DIR = os.path.join(DIR_BASE, "Nivelación de Canastas HCB")
ALIMENTOS_PATH = os.path.join(DIR_BASE, "CONTRATOS ALIMENTOS ORGANIZACIONES CAMPESINAS.xlsx")

OUTPUT_PATH = os.path.join(DIR_BASE, "RECONSTRUCCION_METAVERSO_2026_NACIONAL_AUDITADO.xlsx")

## 2. Funciones de Extracción Granular
La extracción se realiza buscando hojas específicas donde residen las fórmulas validadas. 
* **Integrales:** Se busca la hoja `ZONIFICACIÓN- PEDAGOGICO`.
* **HCB:** Se busca la hoja `BASE DE COSTOS HCB`.
* **Alimentos:** Se procesa el archivo nacional de organizaciones campesinas.

In [8]:
def find_sheet_robust(xl_names, search_terms):
    """Busca una hoja por términos clave de forma insensible a mayúsculas."""
    for name in xl_names:
        if all(term.upper() in name.upper() for term in search_terms):
            return name
    return None

def extract_all_insumos():
    """Recopila todos los datos frescos de los insumos 2026 (Nacional)."""
    print(">>> Extrayendo datos de Insumos 2026 (Integrales, HCB, Alimentos)...")
    rows = []
    
    # 1. INTEGRALES
    int_files = glob.glob(os.path.join(INTEGRALES_DIR, "**", "*.xls*"), recursive=True)
    for f in int_files:
        if "~$" in f: continue
        xl = pd.ExcelFile(f)
        sh_zon = find_sheet_robust(xl.sheet_names, ['ZONIFICA', 'PEDAGOGICO'])
        if sh_zon:
            df = pd.read_excel(f, sheet_name=sh_zon, skiprows=4)
            for _, r in df.iterrows():
                code = str(r.iloc[6]).split('.')[0]
                if len(code) >= 10:
                    rows.append({'Codigo Unidad Servicio UDS': code, 'Cupos a Programar 2026': r.iloc[8], 
                                 'VALOR SOLO 2026': r.iloc[13], 'SERVICIO\n2026': r.iloc[4], 'ZONA 2026': str(r.iloc[0]), 'Modalidad 2026': 'INTEGRAL'})

    # 2. HCB
    hcb_files = glob.glob(os.path.join(HCB_DIR, "**", "*.xls*"), recursive=True)
    for f in hcb_files:
        if "~$" in f: continue
        xl = pd.ExcelFile(f)
        sh_hcb = find_sheet_robust(xl.sheet_names, ['BASE', 'COSTOS', 'HCB'])
        if sh_hcb:
            df = pd.read_excel(f, sheet_name=sh_hcb, skiprows=1)
            for _, r in df.iterrows():
                code = str(r.iloc[6]).split('.')[0]
                if len(code) >= 10:
                    rows.append({'Codigo Unidad Servicio UDS': code, 'Cupos a Programar 2026': r.iloc[9], 
                                 'VALOR SOLO 2026': r.iloc[-1], 'Modalidad 2026': 'HCB'})

    # 3. ALIMENTOS
    df_alim = pd.read_excel(ALIMENTOS_PATH, sheet_name='ZONIFICACION', skiprows=3)
    for _, r in df_alim.iterrows():
        code = str(r.iloc[7]).split('.')[0]
        if len(code) >= 10:
            rows.append({'Codigo Unidad Servicio UDS': code, 'Cupos a Programar 2026': r.iloc[13], 
                         'VALOR SOLO 2026': r.iloc[20], 'SERVICIO\n2026': r.iloc[9], 
                         'CONCEPTO DEFINITIVO': r.iloc[14], 'NIT CONTRATISTA 2026': str(r.iloc[16]).split('.')[0], 
                         'CONTRATISTA 2026': r.iloc[15], 'Modalidad 2026': 'ALIMENTOS'})
            
    return pd.DataFrame(rows).drop_duplicates(subset=['Codigo Unidad Servicio UDS', 'Modalidad 2026'])

## 3. Lógica de Fusión y Auditoría
Se realiza un `merge` (Outer Join) para comparar qué unidades del Metaverso viejo siguen vigentes en el nuevo plan.

In [9]:
print(">>> Iniciando Fusión Nacional...")
df_old = pd.read_excel(ORIGINAL_METAVERSO, sheet_name='ZonificacionPais')
df_old['Codigo Unidad Servicio UDS'] = df_old['Codigo Unidad Servicio UDS'].astype(str).str.split('.').str[0]
headers_orig = df_old.columns.tolist()

df_new = extract_all_insumos()
df_final = pd.merge(df_old, df_new, on='Codigo Unidad Servicio UDS', how='outer', suffixes=('_old', ''))

old_codes = set(df_old['Codigo Unidad Servicio UDS'])
new_codes = set(df_new['Codigo Unidad Servicio UDS'])

def get_status(row):
    code = row['Codigo Unidad Servicio UDS']
    if code in old_codes and code in new_codes: return "ACTUALIZADO (INSUMO 2026)"
    if code in new_codes: return "NUEVO (SOLO INSUMO 2026)"
    return "HISTORICO (SIN RESPALDO 2026)"

df_final['AUDITORIA_ESTADO_2026'] = df_final.apply(get_status, axis=1)
final_cols = headers_orig + ['AUDITORIA_ESTADO_2026']
df_final[final_cols].to_excel(OUTPUT_PATH, index=False)

>>> Iniciando Fusión Nacional...
>>> Extrayendo datos de Insumos 2026 (Integrales, HCB, Alimentos)...


## 4. Formato Visual Condicional
Se aplican colores para facilitar la revisión gerencial:
* **Verde:** Actualizado exitosamente.
* **Azul:** UDS nueva en el plan.
* **Rojo:** UDS histórica desaparecida en el plan 2026.

In [10]:
print(">>> Aplicando formato visual de auditoría...")
wb = load_workbook(OUTPUT_PATH)
ws = wb.active
fills = {
    'ACTUALIZADO': PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid"),
    'NUEVO': PatternFill(start_color="DDEBF7", end_color="DDEBF7", fill_type="solid"),
    'HISTORICO': PatternFill(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
}

col_idx = len(final_cols)
for r in range(2, ws.max_row + 1):
    status = str(ws.cell(row=r, column=col_idx).value)
    for key, fill in fills.items():
        if key in status:
            for c in range(1, col_idx + 1):
                ws.cell(row=r, column=c).fill = fill
            break
wb.save(OUTPUT_PATH)
print(">>> Proceso Finalizado con éxito.")

>>> Aplicando formato visual de auditoría...
>>> Proceso Finalizado con éxito.
